# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_2_HANDS_ON_SESSION_3_planner_thinking.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





#<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About the Notebook

Hands-on: planner **thinking vs no-thinking** + **dense vs MoE** (OpenRouter)

1. Run the **planning** step **with** and **without** thinking enabled (via `extra_body`, and via model choice), and compare **latency**, **reasoning tokens**, and **plan quality**.
2. Run the same workflow with a **dense** planner and a **smaller MoE** planner, and compare behaviour.

This notebook uses [OpenRouter](https://openrouter.ai/docs/quickstart) through its **OpenAI-compatible** `chat.completions` endpoint, so the only things that differ from any other OpenAI-compatible provider are the base URL, the key, and the model slugs. Browse slugs at [openrouter.ai/models](https://openrouter.ai/models).

**API key:** create one at [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys), then put `OPENROUTER_API_KEY` in your `.env` (or paste it when prompted).

| Role | Model slug | Why |
|------|------------|-----|
| Dense planner | `qwen/qwen3.6-27b` | 27B dense, hybrid thinking (can be switched off) |
| MoE planner | `qwen/qwen3.6-35b-a3b` | 35B total / 3B active, same family |
| Thinking planner | `qwen/qwen3-max-thinking` | reasoning-first flagship |
| Executor (fixed) | `openai/gpt-5.4-nano` | cheap second node, held constant |

**No mocking** - cells call the real API when you run them.

## Why `extra_body` matters on OpenRouter

The OpenAI SDK has no first-class field for thinking, so provider-specific parameters ride along in **`extra_body`**, which the SDK merges into the request JSON. On OpenRouter the relevant key is a single normalised [`reasoning`](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens) object, and OpenRouter translates it into whatever the underlying provider actually wants (`thinking_budget` for Qwen, `budget_tokens` for Anthropic, `reasoning_effort` for OpenAI, `thinkingLevel` for Gemini).

```python
extra_body={"reasoning": {"effort": "medium"}}                 # thinking on
extra_body={"reasoning": {"enabled": False}}                   # thinking off
extra_body={"reasoning": {"max_tokens": 2000}}                 # explicit thinking budget
extra_body={"reasoning": {"effort": "high", "exclude": True}}  # think, but hide the trace
```

| Key | Values | Notes |
|-----|--------|-------|
| `effort` | `max`, `xhigh`, `high`, `medium`, `low`, `minimal`, `none` | roughly 95 / 95 / 80 / 50 / 20 / 10 % of `max_tokens`; `none` disables thinking |
| `max_tokens` | int | explicit budget - use `effort` **or** `max_tokens`, never both |
| `enabled` | bool | `true` = thinking at the provider default, `false` = off |
| `exclude` | bool | model still thinks, but the trace is not returned |

### Coming from Nebius

| Nebius Token Factory | OpenRouter |
|----------------------|------------|
| `NEBIUS_API_KEY` | `OPENROUTER_API_KEY` |
| `https://api.tokenfactory.nebius.com/v1` | `https://openrouter.ai/api/v1` |
| catalog id, e.g. `Qwen/Qwen3-Next-80B-A3B-Thinking-fast` | slug, e.g. `qwen/qwen3-max-thinking` |
| free-form `extra_body`, keys differ per model card | one normalised `extra_body={"reasoning": {...}}` |
| pick a separate `-Thinking` checkpoint to get thinking | one hybrid model plus `reasoning.enabled` |
| `message.reasoning_content` | `message.reasoning` (plus `reasoning_details`) |

Two things to watch:

- **Reasoning tokens are output tokens.** They are billed as output and they count against `max_tokens`, so a thinking planner needs a bigger budget or it gets truncated before it reaches the JSON. That is why `OPENROUTER_PLANNER_MAX_TOKENS` defaults to `8192` here.
- **Not every model accepts every key.** A model with mandatory reasoning rejects `effort: "none"`; a non-reasoning model rejects the block entirely. The `chat_create` wrapper below retries once without `extra_body` instead of failing the run.

Latency usually **rises** when the model deliberates before emitting the final JSON. Quality may improve for **decomposition** tasks - measure both wall time and the structural score below.

## Dependencies

In [1]:
%pip install -q "langgraph>=0.2" "openai>=1.40" "python-dotenv>=1.0" "pydantic>=2"

## Configuration

| Variable | Purpose | Default |
|----------|---------|---------|
| `OPENROUTER_API_KEY` | Bearer token | required |
| `OPENROUTER_BASE_URL` | API root | `https://openrouter.ai/api/v1` |
| `OPENROUTER_PLANNER_MODEL_DENSE` | Dense planner | `qwen/qwen3.6-27b` |
| `OPENROUTER_PLANNER_MODEL_MOE` | Smaller MoE planner for the A/B | `qwen/qwen3.6-35b-a3b` |
| `OPENROUTER_PLANNER_MODEL_THINKING` | Reasoning-first planner | `qwen/qwen3-max-thinking` |
| `OPENROUTER_EXECUTOR_MODEL` | Cheap second node, fixed across experiments | `openai/gpt-5.4-nano` |
| `OPENROUTER_PLANNER_REASONING_EFFORT` | Effort used by the "thinking on" run | `medium` |
| `OPENROUTER_PLANNER_EXTRA_BODY_JSON` | Raw JSON that overrides the planner `extra_body` | unset |
| `OPENROUTER_EXECUTOR_EXTRA_BODY_JSON` | Raw JSON that overrides the executor `extra_body` | thinking off |
| `OPENROUTER_PLANNER_MAX_TOKENS` | Planner completion budget (thinking eats into it) | `8192` |
| `OPENROUTER_USAGE_ACCOUNTING` | `1` adds cost + reasoning-token accounting to `usage` | `1` |
| `OPENROUTER_DEMO_TOPIC` | Objective handed to the planner | evaluation-plan prompt |

In [2]:
from dotenv import load_dotenv
import os


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1").strip()

# Colab / no-.env fallback: ask once here instead of failing later inside a graph node.
if not OPENROUTER_API_KEY:
    import getpass

    OPENROUTER_API_KEY = getpass.getpass("OPENROUTER_API_KEY: ").strip()
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print("Base URL:", OPENROUTER_BASE_URL)
print("API key loaded:", bool(OPENROUTER_API_KEY))

Base URL: https://openrouter.ai/api/v1
API key loaded: True


In [3]:
from __future__ import annotations

import json
import os
import re
import time
from typing import Any, NotRequired, TypedDict

from langgraph.graph import END, START, StateGraph
from openai import BadRequestError, OpenAI
from pydantic import BaseModel, Field

OPENROUTER_BASE_URL = os.getenv(
    "OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"
).strip().rstrip("/")

# --- Models (OpenRouter slugs; see https://openrouter.ai/models) ---------------
PLANNER_MODEL_DENSE = os.getenv(
    "OPENROUTER_PLANNER_MODEL_DENSE",
    "qwen/qwen3.6-27b",  # dense 27B, hybrid thinking
).strip()
PLANNER_MODEL_MOE = os.getenv(
    "OPENROUTER_PLANNER_MODEL_MOE",
    "qwen/qwen3.6-35b-a3b",  # MoE 35B total / 3B active
).strip()
PLANNER_MODEL_THINKING = os.getenv(
    "OPENROUTER_PLANNER_MODEL_THINKING",
    "qwen/qwen3-max-thinking",  # reasoning-first flagship
).strip()
EXECUTOR_MODEL = os.getenv(
    "OPENROUTER_EXECUTOR_MODEL",
    "openai/gpt-5.4-nano",  # cheap, fixed across experiments
).strip()

# Reasoning tokens are billed as OUTPUT tokens, so a thinking planner needs headroom.
PLANNER_MAX_TOKENS = int(os.getenv("OPENROUTER_PLANNER_MAX_TOKENS", "8192"))


# --- extra_body -> the OpenRouter `reasoning` object ---------------------------
def reasoning_body(
    effort: str | None = None,
    max_tokens: int | None = None,
    *,
    enabled: bool | None = None,
    exclude: bool = False,
) -> dict[str, Any]:
    """Build the `reasoning` block that OpenRouter expects inside `extra_body`.

    effort     : "max" | "xhigh" | "high" | "medium" | "low" | "minimal" | "none"
                 ("none" disables thinking; roughly 95/95/80/50/20/10 % of max_tokens)
    max_tokens : explicit thinking budget (Anthropic style; Qwen maps it to thinking_budget)
    enabled    : True  -> thinking on at the provider default effort
                 False -> thinking off
    exclude    : True  -> model still thinks, but the trace is not returned

    Pass `effort` OR `max_tokens`, never both.
    """
    if effort is not None and max_tokens is not None:
        raise ValueError("Pass effort OR max_tokens, not both.")
    cfg: dict[str, Any] = {}
    if effort is not None:
        cfg["effort"] = effort
    if max_tokens is not None:
        cfg["max_tokens"] = int(max_tokens)
    if enabled is not None:
        cfg["enabled"] = bool(enabled)
    if exclude:
        cfg["exclude"] = True
    return {"reasoning": cfg} if cfg else {}


# Two named policies used by the experiments below.
THINKING_OFF: dict[str, Any] = reasoning_body(enabled=False)
THINKING_ON: dict[str, Any] = reasoning_body(
    effort=os.getenv("OPENROUTER_PLANNER_REASONING_EFFORT", "medium")
)

# Optional raw override, e.g. OPENROUTER_PLANNER_EXTRA_BODY_JSON='{"reasoning":{"max_tokens":2000}}'
_extra = os.getenv("OPENROUTER_PLANNER_EXTRA_BODY_JSON", "").strip()
PLANNER_EXTRA_BODY: dict[str, Any] = json.loads(_extra) if _extra else THINKING_ON

# The executor is a GPT-5.x model, so it can think too. Keep it off: we want the
# planner to be the only thing that changes between runs.
_extra_exec = os.getenv("OPENROUTER_EXECUTOR_EXTRA_BODY_JSON", "").strip()
EXECUTOR_EXTRA_BODY: dict[str, Any] = (
    json.loads(_extra_exec) if _extra_exec else reasoning_body(enabled=False)
)

# Usage accounting returns cost + reasoning_tokens in `usage` (set to 0 to disable).
USAGE_ACCOUNTING: dict[str, Any] = (
    {"usage": {"include": True}}
    if os.getenv("OPENROUTER_USAGE_ACCOUNTING", "1") == "1"
    else {}
)

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers={
        # Optional attribution headers; they put your app on the OpenRouter leaderboards.
        "HTTP-Referer": os.getenv(
            "OPENROUTER_SITE_URL", "https://github.com/Nicolepcx/ORM_AI_Agents_Bootcamp"
        ),
        "X-Title": os.getenv("OPENROUTER_APP_TITLE", "AI Agents Bootcamp - planner thinking"),
    },
)


def merge_extra(*bodies: dict[str, Any] | None) -> dict[str, Any]:
    """Shallow-merge extra_body dicts, one level deep for nested objects."""
    out: dict[str, Any] = {}
    for body in bodies:
        for key, val in (body or {}).items():
            if val is None:
                continue
            if isinstance(val, dict) and isinstance(out.get(key), dict):
                out[key] = {**out[key], **val}
            else:
                out[key] = val
    return out


def chat_create(**kwargs: Any):
    """chat.completions.create + two guardrails.

    1. Not every model accepts every reasoning key (e.g. a model with mandatory
       reasoning rejects effort="none"). On a 400 we retry once without extra_body
       instead of killing the run.
    2. OpenRouter can return 200 with an `error` payload and no choices.
    """
    try:
        resp = client.chat.completions.create(**kwargs)
    except BadRequestError as err:
        if not kwargs.get("extra_body"):
            raise
        print(f"[warn] {kwargs.get('model')} rejected extra_body={kwargs['extra_body']}: {err}")
        print("[warn] retrying without extra_body")
        kwargs = {k: v for k, v in kwargs.items() if k != "extra_body"}
        resp = client.chat.completions.create(**kwargs)
    if not getattr(resp, "choices", None):
        raise RuntimeError(f"No choices returned: {getattr(resp, 'error', resp)}")
    return resp


print("Base URL:", OPENROUTER_BASE_URL)
print("Dense planner:", PLANNER_MODEL_DENSE)
print("MoE planner:", PLANNER_MODEL_MOE)
print("Thinking planner:", PLANNER_MODEL_THINKING)
print("Executor:", EXECUTOR_MODEL)
print("Planner max_tokens:", PLANNER_MAX_TOKENS)
print("Thinking OFF extra_body:", THINKING_OFF)
print("Thinking ON  extra_body:", PLANNER_EXTRA_BODY)
print("Executor extra_body:", EXECUTOR_EXTRA_BODY)

Base URL: https://openrouter.ai/api/v1
Dense planner: qwen/qwen3.6-27b
MoE planner: qwen/qwen3.6-35b-a3b
Thinking planner: qwen/qwen3-max-thinking
Executor: openai/gpt-5.4-nano
Planner max_tokens: 8192
Thinking OFF extra_body: {'reasoning': {'enabled': False}}
Thinking ON  extra_body: {'reasoning': {'effort': 'medium'}}
Executor extra_body: {'reasoning': {'enabled': False}}


### Which knobs does each model actually accept?

`GET /api/v1/models` reports a `reasoning` block per model: the supported effort levels, whether thinking is on by default, and whether it is mandatory. This is the OpenRouter equivalent of reading a Nebius model card to find out which `extra_body` keys exist. Optional, but it explains any `[warn] ... rejected extra_body` messages you may see later.

In [4]:
import httpx  # ships with the openai SDK

_CATALOG: dict[str, dict] = {}


def _catalog() -> dict[str, dict]:
    """Cache GET /api/v1/models once per session."""
    if not _CATALOG:
        r = httpx.get(f"{OPENROUTER_BASE_URL}/models", timeout=60)
        r.raise_for_status()
        for m in r.json().get("data", []):
            _CATALOG[m.get("id", "")] = m
    return _CATALOG


def describe_reasoning(model_id: str) -> str:
    """What reasoning knobs does this model actually accept?"""
    m = _catalog().get(model_id)
    if m is None:
        return f"{model_id}: NOT in the catalog (check the slug)"
    r = m.get("reasoning")
    if not r:
        return f"{model_id}: no reasoning block -> non-reasoning model, omit extra_body"
    return (
        f"{model_id}: efforts={r.get('supported_efforts')} "
        f"default_effort={r.get('default_effort')} "
        f"on_by_default={r.get('default_enabled')} "
        f"supports_max_tokens={r.get('supports_max_tokens')} "
        f"mandatory={r.get('mandatory')}"
    )


try:
    for _m in (PLANNER_MODEL_DENSE, PLANNER_MODEL_MOE, PLANNER_MODEL_THINKING, EXECUTOR_MODEL):
        print(describe_reasoning(_m))
except Exception as exc:  # offline / rate limited -> not fatal, the experiments still run
    print("(Catalog lookup skipped)", exc)

qwen/qwen3.6-27b: efforts=None default_effort=None on_by_default=True supports_max_tokens=None mandatory=False
qwen/qwen3.6-35b-a3b: efforts=None default_effort=None on_by_default=True supports_max_tokens=None mandatory=False
qwen/qwen3-max-thinking: efforts=None default_effort=None on_by_default=None supports_max_tokens=None mandatory=False
openai/gpt-5.4-nano: efforts=['xhigh', 'high', 'medium', 'low', 'none'] default_effort=medium on_by_default=False supports_max_tokens=None mandatory=False


## Plan schema + prompts

The planner should return **JSON**, possibly after a thinking block. The code reads `message.content` first and falls back to `reasoning` / `reasoning_content` / `reasoning_details` when the visible answer is empty, strips leftover `<think>` fences, then parses. If that still fails, a **repair** call uses the **user objective** plus the raw output, so `steps` stay real plan steps rather than meta-instructions about JSON.

In [5]:
from typing import Any

PLANNER_SYSTEM = """You are a planning component inside a larger workflow.
Given a user objective, output ONLY valid JSON (no markdown fences) with this shape:
{"goal_restated": str, "steps": [str, ...], "checklist": [str, ...], "risks": [str, ...]}
If you use internal reasoning or thinking tags, the final answer must still end with this JSON only.
Constraints:
- steps: 4-8 concrete ordered actions (each one line) for the USER objective.
- checklist: 3-6 verifiable completion checks.
- risks: 2-5 plausible failure modes.
JSON rules (critical): use straight ASCII double quotes only; escape internal " as \\"; no trailing commas;
no raw line breaks inside strings; avoid apostrophes in strings (write cannot not can't).
Do not add keys. Do not wrap in ```."""


class Plan(BaseModel):
    goal_restated: str = Field(..., min_length=8)
    steps: list[str] = Field(..., min_length=4, max_length=10)
    checklist: list[str] = Field(..., min_length=3, max_length=8)
    risks: list[str] = Field(..., min_length=2, max_length=8)


def _strip_json_fence(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        lines = s.split("\n")
        if lines:
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        s = "\n".join(lines).strip()
    return s


_THINK_BLOCK = re.compile(r"<think>[\s\S]*?</think>", re.I)


def _strip_think_tags(s: str) -> str:
    """Drop Qwen / DeepSeek style think fences that some providers inline into content."""
    s = _THINK_BLOCK.sub("", s or "")
    if "</think>" in s:  # unclosed opening tag: keep everything after the close
        s = s.split("</think>", 1)[-1]
    return s.strip()


def _assistant_parts(msg: Any) -> tuple[str, str]:
    """Split an OpenRouter message into (visible content, reasoning trace).

    OpenRouter returns thinking in `message.reasoning` (alias `reasoning_content`)
    or, for structured traces, in `message.reasoning_details`. Some models leave
    `content` empty and put everything in the reasoning fields, so we keep both.
    """
    content_chunks: list[str] = []
    c = getattr(msg, "content", None)
    if isinstance(c, str) and c.strip():
        content_chunks.append(c.strip())
    elif isinstance(c, list):
        for part in c:
            if isinstance(part, dict) and part.get("type") == "text":
                t = (part.get("text") or "").strip()
                if t:
                    content_chunks.append(t)
            elif isinstance(part, str) and part.strip():
                content_chunks.append(part.strip())

    reasoning = ""
    for attr in ("reasoning", "reasoning_content"):
        r = getattr(msg, attr, None)
        if r:
            reasoning = str(r).strip()
            break
    if not reasoning:
        details = getattr(msg, "reasoning_details", None) or []
        texts: list[str] = []
        for d in details:
            if not isinstance(d, dict):
                d = d.model_dump() if hasattr(d, "model_dump") else {}
            t = d.get("text") or d.get("summary")
            if t:
                texts.append(str(t))
        reasoning = "\n".join(texts).strip()

    return "\n\n".join(content_chunks).strip(), reasoning


def _extract_first_json_object(s: str) -> str:
    s = _strip_think_tags(_strip_json_fence(s))
    i = s.find("{")
    if i == -1:
        return s
    depth = 0
    for j, ch in enumerate(s[i:], start=i):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[i : j + 1]
    return s


_TRAILING_COMMA = re.compile(r",(\s*[}\]])")


def _relax_json_text(blob: str) -> str:
    s = blob.strip()
    s = (
        s.replace(chr(0x201C), '"')
        .replace(chr(0x201D), '"')
        .replace(chr(0x2019), "'")
    )
    s = _TRAILING_COMMA.sub(r"\1", s)
    return s


def parse_plan_from_model_text(raw: str) -> Plan:
    """Parse planner output: strict JSON first, then light repair + raw_decode scan."""
    raw = (raw or "").strip()
    if not raw:
        raise ValueError("Empty planner text (no content or reasoning to parse).")
    payload = _extract_first_json_object(raw)
    if not (payload or "").strip():
        raise ValueError("No JSON object found in planner text.")
    errors: list[str] = []
    for attempt, text in enumerate((payload, _relax_json_text(payload))):
        try:
            return Plan.model_validate_json(text)
        except Exception as e:
            errors.append(f"validate_json[{attempt}]: {e}")
        try:
            data = json.loads(text)
            return Plan.model_validate(data)
        except Exception as e:
            errors.append(f"json.loads[{attempt}]: {e}")
    dec = json.JSONDecoder()
    s = _relax_json_text(payload)
    i = 0
    while True:
        j = s.find("{", i)
        if j == -1:
            break
        try:
            obj, end = dec.raw_decode(s, j)
            return Plan.model_validate(obj)
        except Exception as e:
            errors.append(f"raw_decode@{j}: {e}")
            i = j + 1
    raise ValueError("Could not parse Plan JSON. Last errors: " + " | ".join(errors[-6:]))


def plan_quality_score(p: Plan) -> dict[str, Any]:
    """Cheap automatic signals (not ground-truth quality)."""
    steps_lower = [x.strip().lower() for x in p.steps if x.strip()]
    dup = len(steps_lower) - len(set(steps_lower))
    return {
        "n_steps": len(p.steps),
        "n_checklist": len(p.checklist),
        "n_risks": len(p.risks),
        "duplicate_steps": dup,
        "avg_step_len": round(sum(len(s) for s in p.steps) / max(len(p.steps), 1), 1),
    }


def usage_dict(resp: Any) -> dict[str, Any] | None:
    u = getattr(resp, "usage", None)
    if u is None:
        return None
    return u.model_dump() if hasattr(u, "model_dump") else dict(u)


def usage_highlights(usage: dict[str, Any] | None) -> dict[str, Any]:
    """Pull the two numbers that make thinking visible: reasoning tokens and cost."""
    if not usage:
        return {}
    details = usage.get("completion_tokens_details") or {}
    reasoning_tokens = details.get("reasoning_tokens") if isinstance(details, dict) else None
    return {
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "reasoning_tokens": reasoning_tokens,
        "cost_usd": usage.get("cost"),
    }


REPAIR_SYSTEM = """You output ONLY one JSON object (no markdown).
Keys exactly: goal_restated, steps, checklist, risks.
- goal_restated: one sentence restating the USER objective from the prompt.
- steps: 4-8 strings. Each step must be a real action for that objective (e.g. define metrics, run pilot).
  Do NOT put meta text in steps (no "fix JSON", "remove markdown", or "output only").
- checklist: 3-6 verifiable checks. risks: 2-5 strings.
Use ASCII double quotes and valid JSON escapes."""


def call_planner(
    topic: str,
    *,
    model: str,
    extra_body: dict[str, Any] | None = None,
    temperature: float = 0.2,
    max_tokens: int | None = None,
) -> tuple[Plan, dict[str, Any]]:
    """Returns parsed plan + metadata (latency, usage, reasoning head)."""
    mt = max_tokens if max_tokens is not None else PLANNER_MAX_TOKENS
    body = merge_extra(extra_body, USAGE_ACCOUNTING)
    t0 = time.perf_counter()
    create_kw: dict[str, Any] = {
        "model": model,
        "temperature": temperature,
        "max_tokens": mt,
        "messages": [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": topic},
        ],
    }
    if body:
        create_kw["extra_body"] = body
    resp = chat_create(**create_kw)
    msg = resp.choices[0].message
    content, reasoning = _assistant_parts(msg)

    # Try the visible answer first; only fall back to the thinking trace if the
    # model put its JSON in there (happens when max_tokens cuts the answer short).
    candidates = [t for t in (content, f"{content}\n\n{reasoning}".strip(), reasoning) if t.strip()]
    meta: dict[str, Any] = {"json_repair_used": False}
    plan: Plan | None = None
    for cand in candidates:
        try:
            plan = parse_plan_from_model_text(cand)
            break
        except ValueError:
            continue

    if plan is None:
        raw = content or reasoning
        blob = raw if raw.strip() else str(msg)[:12000]
        fix_user = (
            f"USER OBJECTIVE:\n{topic}\n\n"
            f"PLANNER MODEL OUTPUT (may be broken or mixed with thinking):\n{blob[:12000]}"
        )
        fix_kw: dict[str, Any] = {
            "model": EXECUTOR_MODEL,
            "temperature": 0,
            "max_tokens": 2048,
            "messages": [
                {"role": "system", "content": REPAIR_SYSTEM},
                {"role": "user", "content": fix_user},
            ],
        }
        fix_body = merge_extra(EXECUTOR_EXTRA_BODY, USAGE_ACCOUNTING)
        if fix_body:
            fix_kw["extra_body"] = fix_body
        fix = chat_create(**fix_kw)
        raw2, reasoning2 = _assistant_parts(fix.choices[0].message)
        try:
            plan = parse_plan_from_model_text(raw2 or reasoning2)
        except ValueError as err2:
            hint = (blob[:500] + "...") if blob else "(empty content and reasoning)"
            raise ValueError(
                "Planner output was not valid JSON even after repair pass. "
                f"Planner text (head): {hint}"
            ) from err2
        meta["json_repair_used"] = True

    wall_s = time.perf_counter() - t0
    usage = usage_dict(resp)
    meta.update(
        {
            "wall_s": round(wall_s, 4),
            "usage": usage,
            "usage_head": usage_highlights(usage),
            "finish_reason": getattr(resp.choices[0], "finish_reason", None),
            "raw_head": content[:400] + ("..." if len(content) > 400 else ""),
            "reasoning_head": (reasoning[:400] + "...") if reasoning else None,
            "reasoning_chars": len(reasoning),
        }
    )
    return plan, meta

## LangGraph workflow: `plan` -> `execute`

The **executor** is deliberately simple: it turns the structured plan into a short Markdown summary using a **fixed** small model, with its own thinking switched off, so differences mostly reflect the **planner**.

In [6]:
class WorkflowState(TypedDict):
    topic: str
    planner_model: str
    planner_label: str
    planner_extra_body: NotRequired[dict[str, Any]]
    plan: NotRequired[dict]
    plan_meta: NotRequired[dict]
    execution: NotRequired[str]
    execute_meta: NotRequired[dict]
    timings: NotRequired[dict[str, float]]


def node_plan(state: WorkflowState) -> dict:
    extra = dict(state.get("planner_extra_body") or {})
    plan, meta = call_planner(
        state["topic"],
        model=state["planner_model"],
        extra_body=extra,
    )
    tim = dict(state.get("timings") or {})
    tim["plan_llm_s"] = float(meta["wall_s"])
    return {
        "plan": plan.model_dump(),
        "plan_meta": meta,
        "timings": tim,
    }


def node_execute(state: WorkflowState) -> dict:
    t0 = time.perf_counter()
    user = (
        f"Planner label: {state['planner_label']}\n"
        f"Objective:\n{state['topic']}\n\n"
        f"Structured plan (JSON):\n{json.dumps(state['plan'], ensure_ascii=False)}\n\n"
        "Write a concise Markdown brief: ## Summary, ## Recommended order, ## Watchouts. "
        "Do not change the plan JSON; interpret it."
    )
    exec_kw: dict[str, Any] = {
        "model": EXECUTOR_MODEL,
        "temperature": 0.3,
        "max_tokens": 900,
        "messages": [{"role": "user", "content": user}],
    }
    # Executor thinking stays OFF so that differences trace back to the planner.
    exec_body = merge_extra(EXECUTOR_EXTRA_BODY, USAGE_ACCOUNTING)
    if exec_body:
        exec_kw["extra_body"] = exec_body
    resp = chat_create(**exec_kw)
    wall_s = time.perf_counter() - t0
    text = (resp.choices[0].message.content or "").strip()
    usage = usage_dict(resp)
    tim = dict(state.get("timings") or {})
    tim["execute_llm_s"] = round(wall_s, 4)
    return {
        "execution": text,
        "execute_meta": {
            "wall_s": tim["execute_llm_s"],
            "usage": usage,
            "usage_head": usage_highlights(usage),
        },
        "timings": tim,
    }


g = StateGraph(WorkflowState)
g.add_node("plan", node_plan)
g.add_node("execute", node_execute)
g.add_edge(START, "plan")
g.add_edge("plan", "execute")
g.add_edge("execute", END)
app = g.compile()

try:
    print(app.get_graph().draw_mermaid())
except Exception as exc:
    print("(Mermaid skipped)", exc)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	execute(execute)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	plan --> execute;
	execute --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Experiment A - same dense planner: thinking **off** vs **on** vs a **reasoning-first model**

1. **No-thinking baseline** - `qwen/qwen3.6-27b` with `{"reasoning": {"enabled": False}}`. Note that "no `extra_body`" is *not* the same thing on a hybrid model: leave it out and the model may think anyway.
2. **Same model, thinking on** - `{"reasoning": {"effort": "medium"}}` (change with `OPENROUTER_PLANNER_REASONING_EFFORT`).
3. **Reasoning-first model** - `qwen/qwen3-max-thinking` at provider defaults.

Watch `reasoning_tokens` and `cost_usd` in the `usage` line, not just wall time.

In [7]:
TOPIC = os.getenv(
    "OPENROUTER_DEMO_TOPIC",
    "Design a short evaluation plan for a customer-support agent before production rollout.",
)


def run_labeled(label: str, planner_model: str, extra: dict[str, Any] | None) -> dict:
    t_wall0 = time.perf_counter()
    out = app.invoke(
        {
            "topic": TOPIC,
            "planner_model": planner_model,
            "planner_label": label,
            "planner_extra_body": extra or {},
        }
    )
    wall = time.perf_counter() - t_wall0
    plan = Plan.model_validate(out["plan"])
    score = plan_quality_score(plan)
    return {
        "label": label,
        "planner_model": planner_model,
        "extra_body": extra or {},
        "wall_total_s": round(wall, 4),
        "timings": out.get("timings") or {},
        "quality": score,
        "plan": out["plan"],
        "plan_meta": out.get("plan_meta") or {},
        "execution_excerpt": (out.get("execution") or "")[:1500],
    }


rows_a = [
    # 1. same model, thinking explicitly OFF  -> {"reasoning": {"enabled": False}}
    run_labeled("dense_thinking_off", PLANNER_MODEL_DENSE, THINKING_OFF),
    # 2. same model, thinking ON              -> {"reasoning": {"effort": "medium"}}
    run_labeled("dense_thinking_on", PLANNER_MODEL_DENSE, PLANNER_EXTRA_BODY),
    # 3. reasoning-first model, provider defaults (it always thinks)
    run_labeled("thinking_model", PLANNER_MODEL_THINKING, None),
]

print("Topic:", TOPIC)
for r in rows_a:
    print("\n===", r["label"], "===")
    print("planner:", r["planner_model"], "| extra_body:", r["extra_body"] or "{}")
    print("wall_total_s:", r["wall_total_s"], "| node timings:", r["timings"])
    print("usage:", r["plan_meta"].get("usage_head") or "(no usage returned)")
    print("reasoning_chars:", r["plan_meta"].get("reasoning_chars"))
    if r["plan_meta"].get("json_repair_used"):
        print("(Planner JSON needed a repair pass via executor model - adds latency.)")
    print("quality:", r["quality"])
    if r["plan_meta"].get("reasoning_head"):
        print("reasoning_head:", r["plan_meta"]["reasoning_head"])
    print("execution_excerpt:\n", r["execution_excerpt"])

Topic: Design a short evaluation plan for a customer-support agent before production rollout.

=== dense_thinking_off ===
planner: qwen/qwen3.6-27b | extra_body: {'reasoning': {'enabled': False}}
wall_total_s: 11.3785 | node timings: {'plan_llm_s': 7.6143, 'execute_llm_s': 3.7466}
usage: {'prompt_tokens': 220, 'completion_tokens': 250, 'reasoning_tokens': 0, 'cost_usd': 0.00066358}
reasoning_chars: 0
quality: {'n_steps': 6, 'n_checklist': 5, 'n_risks': 4, 'duplicate_steps': 0, 'avg_step_len': 96.5}
execution_excerpt:
 ## Summary
A short, structured pre-rollout evaluation to determine whether the customer-support agent is ready for production. It combines quantitative success metrics (e.g., first response time, resolution rate, CSAT), realistic test coverage (common + edge cases), live supervised shadowing with real-time feedback, a blind historical-data check for accuracy/tone, and qualitative review (communication and empathy), culminating in a go/no-go readiness report.

## Recommend

## Experiment B - **dense** vs **MoE** planner (same reasoning policy)

Hold the executor and the reasoning policy fixed, and change only the planner: a **dense** 27B against a **35B MoE with 3B active parameters**. If a slug is unavailable in your account, override `OPENROUTER_PLANNER_MODEL_MOE`.

In [8]:
# One shared reasoning policy for both planners. Flip to THINKING_ON and re-run
# to see whether the MoE (3B active) closes the gap when it is allowed to think.
POLICY_B = THINKING_OFF

rows_b = [
    run_labeled("planner_dense", PLANNER_MODEL_DENSE, POLICY_B),
    run_labeled("planner_moe", PLANNER_MODEL_MOE, POLICY_B),
]

for r in rows_b:
    print("\n===", r["label"], "===")
    print("planner:", r["planner_model"], "| extra_body:", r["extra_body"] or "{}")
    print("wall_total_s:", r["wall_total_s"], "| plan node:", r["timings"].get("plan_llm_s"))
    print("usage:", r["plan_meta"].get("usage_head") or "(no usage returned)")
    if r["plan_meta"].get("json_repair_used"):
        print("(JSON repair pass used.)")
    print("quality:", r["quality"])

print("\n--- Side-by-side ---")
print(
    json.dumps(
        [
            {
                "label": r["label"],
                "model": r["planner_model"],
                "wall_total_s": r["wall_total_s"],
                "plan_s": r["timings"].get("plan_llm_s"),
                "reasoning_tokens": (r["plan_meta"].get("usage_head") or {}).get("reasoning_tokens"),
                "cost_usd": (r["plan_meta"].get("usage_head") or {}).get("cost_usd"),
                "quality": r["quality"],
            }
            for r in rows_b
        ],
        indent=2,
    )
)


=== planner_dense ===
planner: qwen/qwen3.6-27b | extra_body: {'reasoning': {'enabled': False}}
wall_total_s: 6.9415 | plan node: 4.2375
usage: {'prompt_tokens': 220, 'completion_tokens': 294, 'reasoning_tokens': 0, 'cost_usd': 0.001027}
quality: {'n_steps': 6, 'n_checklist': 5, 'n_risks': 4, 'duplicate_steps': 0, 'avg_step_len': 97.8}

=== planner_moe ===
planner: qwen/qwen3.6-35b-a3b | extra_body: {'reasoning': {'enabled': False}}
wall_total_s: 5.6811 | plan node: 2.8101
usage: {'prompt_tokens': 220, 'completion_tokens': 301, 'reasoning_tokens': 0, 'cost_usd': 0.000334}
quality: {'n_steps': 6, 'n_checklist': 5, 'n_risks': 4, 'duplicate_steps': 0, 'avg_step_len': 93.0}

--- Side-by-side ---
[
  {
    "label": "planner_dense",
    "model": "qwen/qwen3.6-27b",
    "wall_total_s": 6.9415,
    "plan_s": 4.2375,
    "reasoning_tokens": 0,
    "cost_usd": 0.001027,
    "quality": {
      "n_steps": 6,
      "n_checklist": 5,
      "n_risks": 4,
      "duplicate_steps": 0,
      "avg_step_l

## What to take away

- On OpenRouter, thinking is one normalised parameter: `extra_body={"reasoning": {...}}`. You do not need per-model magic keys, and you can swap `qwen/...` for `anthropic/...` or `google/...` without touching the call site.
- **Thinking off is a choice you have to make explicitly** on hybrid models. Omitting `extra_body` gives you the provider default, which is often thinking **on**.
- Reasoning tokens are output tokens: they show up in latency, in `usage.completion_tokens`, and in cost. Budget `max_tokens` accordingly, or the JSON gets truncated mid-plan.
- Active parameters, not total parameters, drive MoE latency - which is why a 35B-A3B model can be quicker than a 27B dense one while still trailing it on decomposition quality.